# TechMart — Data Quality Notebook

**Scenario:** TechMart (E-commerce)  
**Project:** Group project — Big Data Architecture  

## Why this notebook

This notebook implements the data quality gate for the TechMart pipeline.
12 carefully chosen checks covering all five quality dimensions:

| Dimension      | Checks here              |
|----------------|---------------------------|
| Completeness   | O-1, O-2, O-3            |
| Uniqueness     | O-4                      |
| Validity       | O-5, O-6, O-7, O-8       |
| Consistency    | O-9, O-10                |
| Timeliness     | O-11, O-12               |


## 1. Spark + PyDeequ setup

In [1]:
import os

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TechMart-DQ")
    .config("spark.jars.packages", "com.amazon.deequ:deequ:2.0.7-spark-3.5")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")

os.environ["SPARK_VERSION"] = spark.version

Spark version: 3.5.0


In [2]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult
from pydeequ.analyzers import (
    AnalysisRunner,
    Size,
    Completeness,
    ApproxCountDistinct
)

print("PyDeequ ready")

PyDeequ ready


## 2. Load ordersWe use an **explicit schema** rather than `inferSchema` so a malformed CSV row cannot silently mis-cast a column.

In [3]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)

orders_schema = StructType([
    StructField("order_id",       StringType(),    True),
    StructField("customer_id",    StringType(),    True),
    StructField("product_id",     StringType(),    True),
    StructField("quantity",       IntegerType(),   True),
    StructField("unit_price",     DoubleType(),    True),
    StructField("total_amount",   DoubleType(),    True),
    StructField("order_date",     TimestampType(), True),
    StructField("status",         StringType(),    True),
    StructField("payment_method", StringType(),    True),
])

orders = spark.read.csv(
    "../data/techmart/orders.csv",
    header=True,
    schema=orders_schema
)

print(f"Orders rows: {orders.count():,}")

orders.show(5, truncate=False)

Orders rows: 58,355
+--------------+-----------+----------+--------+----------+------------+-------------------+---------+--------------+
|order_id      |customer_id|product_id|quantity|unit_price|total_amount|order_date         |status   |payment_method|
+--------------+-----------+----------+--------+----------+------------+-------------------+---------+--------------+
|ORD-0000000001|C0002      |P002      |3       |999.0     |2997.0      |2025-06-18 13:17:00|completed|bank_transfer |
|ORD-0000000002|C0003      |P003      |1       |349.0     |349.0       |2025-10-23 23:24:00|cancelled|credit_card   |
|ORD-0000000003|C0004      |P004      |5       |599.0     |2995.0      |2025-06-14 14:35:00|completed|paypal        |
|ORD-0000000004|C0005      |P005      |2       |139.0     |278.0       |2025-01-31 15:51:00|completed|credit_card   |
|ORD-0000000005|C0006      |P006      |1       |129.0     |129.0       |2025-05-10 05:27:00|completed|credit_card   |
+--------------+-----------+--------

## 3. Quick profileWe run a tiny profiling pass with Deequ analyzers — same metrics will go to InfluxDB later so we can spot drift over time.

In [5]:
from pydeequ.analyzers import (
    AnalysisRunner,
    AnalyzerContext,
    Size,
    Completeness,
    ApproxCountDistinct
)
 
runner = (
    AnalysisRunner(spark)
        .onData(orders)
        .addAnalyzer(Size())
)
 
for c in [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "status",
    "payment_method"
]:
    runner = (
        runner
            .addAnalyzer(Completeness(c))
            .addAnalyzer(ApproxCountDistinct(c))
    )
 
profile = runner.run()
 
prof_df = AnalyzerContext.successMetricsAsDataFrame(
    spark,
    profile
)
 
prof_df.show(truncate=False)
 
orders.describe([
    "quantity",
    "unit_price",
    "total_amount"
]).show()

/usr/local/spark/python/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+-------+--------------+-------------------+-------+
|entity |instance      |name               |value  |
+-------+--------------+-------------------+-------+
|Column |quantity      |Completeness       |1.0    |
|Column |status        |Completeness       |1.0    |
|Column |payment_method|Completeness       |1.0    |
|Column |order_id      |ApproxCountDistinct|62610.0|
|Column |quantity      |ApproxCountDistinct|5.0    |
|Column |customer_id   |Completeness       |1.0    |
|Dataset|*             |Size               |58355.0|
|Column |product_id    |Completeness       |1.0    |
|Column |customer_id   |ApproxCountDistinct|6113.0 |
|Column |order_id      |Completeness       |1.0    |
|Column |payment_method|ApproxCountDistinct|4.0    |
|Column |status        |ApproxCountDistinct|4.0    |
|Column |product_id    |ApproxCountDistinct|20.0   |
+-------+--------------+-------------------+-------+

+-------+------------------+------------------+------------------+
|summary|          quantity|   

## 4. Quality rules — 12 checks across 5 dimensions

| #    | Dimension      | Column          | Rule                                              | Business consequence of failure |
|------|----------------|-----------------|---------------------------------------------------|---------------------------------|
| O-1  | Completeness   | order_id        | not null                                          | Cannot audit; breaks payment reconciliation |
| O-2  | Completeness   | customer_id     | not null                                          | Revenue can't be attributed; recommender corrupts |
| O-3  | Completeness   | product_id      | not null                                          | Inventory delta cannot be applied |
| O-4  | Uniqueness     | order_id        | unique                                            | Double-counted revenue (the Campus-Café bug) |
| O-5  | Range          | quantity        | between 1 and 50                                  | 0 = phantom order; > 50 likely a wholesale leak |
| O-6  | Range          | unit_price      | between 0.01 and 10 000                           | Negative prices = data corruption |
| O-7  | Range          | total_amount    | non-negative                                      | Negative totals corrupt revenue dashboards |
| O-8  | Consistency    | total_amount    | = quantity × unit_price (± 0.01 €)                | Pricing engine drift / coupon bugs |
| O-9  | Validity       | status          | ∈ {completed, cancelled, pending, shipped}        | Unknown states silently dropped by dashboards |
| O-10 | Validity       | payment_method  | ∈ {credit_card, debit_card, paypal, bank_transfer} | Payment-gateway drift |
| O-11 | Range          | order_date      | not in the future                                 | Clock skew / replay events |
| O-12 | Range          | order_date      | not older than 5 years                            | Out-of-policy stale records |

## 5. Build the verification suite

In [6]:
ORDER_STATUSES = [
    "completed",
    "cancelled",
    "pending",
    "shipped"
]

PAYMENT_METHODS = [
    "credit_card",
    "debit_card",
    "paypal",
    "bank_transfer"
]

def in_list_sql(col, values):
    return f"{col} IN (" + ", ".join(f"'{v}'" for v in values) + ")"

orders_check = (
    Check(spark, CheckLevel.Error, "TechMart Orders Quality")

    .isComplete("order_id")                    # O-1
    .isComplete("customer_id")                 # O-2
    .isComplete("product_id")                  # O-3

    .isUnique("order_id")                      # O-4

    .satisfies(
        "quantity BETWEEN 1 AND 50",
        "O-5 quantity in [1,50]"
    )

    .satisfies(
        "unit_price BETWEEN 0.01 AND 10000",
        "O-6 unit_price in (0,10000]"
    )

    .isNonNegative("total_amount")             # O-7

    .satisfies(
        "abs(total_amount - quantity * unit_price) < 0.01",
        "O-8 total = qty * unit_price"
    )

    .satisfies(
        in_list_sql("status", ORDER_STATUSES),
        "O-9 status in allow-list"
    )

    .satisfies(
        in_list_sql("payment_method", PAYMENT_METHODS),
        "O-10 payment_method in allow-list"
    )

    .satisfies(
        "order_date <= current_timestamp()",
        "O-11 order_date not in future"
    )

    .satisfies(
        "order_date >= add_months(current_timestamp(), -60)",
        "O-12 order_date within 5 years"
    )
)

print("Suite built with 12 constraints")

Suite built with 12 constraints


## 6. Run the verification

In [7]:
result = (
    VerificationSuite(spark)
    .onData(orders)
    .addCheck(orders_check)
    .run()
)

result_df = VerificationResult.checkResultsAsDataFrame(
    spark,
    result
)

result_df.show(20, truncate=False)

result_pdf = result_df.toPandas()

+-----------------------+-----------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+
|check                  |check_level|check_status|constraint                                                                                                                                                     |constraint_status|constraint_message|
+-----------------------+-----------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+
|TechMart Orders Quality|Error      |Success     |CompletenessConstraint(Completeness(order_id,None,None))                                                                                                       |Success          |                  |
|TechMar

## 7. Quality score and gateScore = share of constraints that returned `Success`. Gate: green ≥ 90 %, yellow ≥ 70 %, red < 70 %.

In [8]:
total = len(result_pdf)

passed = (
    result_pdf["constraint_status"] == "Success"
).sum()

failed = total - passed

score = round(
    100.0 * passed / total,
    2
) if total else 0.0

gate = (
    "GREEN"
    if score >= 90
    else ("YELLOW" if score >= 70 else "RED")
)

print(f"Total constraints  : {total}")
print(f"Passed             : {passed}")
print(f"Failed             : {failed}")
print(f"Score              : {score} %")
print(f"Gate               : {gate}")

failing = result_pdf[
    result_pdf["constraint_status"] != "Success"
][[
    "constraint",
    "constraint_status",
    "constraint_message"
]]

if len(failing):
    print("\nFailing constraints:")
    print(failing.to_string(index=False))
else:
    print("\nAll constraints passed.")

Total constraints  : 12
Passed             : 12
Failed             : 0
Score              : 100.0 %
Gate               : GREEN

All constraints passed.


## 8. Persist the report (JSON)

In [9]:
import json
import datetime
import os

os.makedirs("../quality_reports", exist_ok=True)

ts = datetime.datetime.utcnow().strftime(
    "%Y%m%dT%H%M%SZ"
)

report_path = (
    f"../quality_reports/techmart_{ts}.json"
)

report = {
    "run_id": ts,
    "scenario": "techmart",
    "team": "techmart",
    "score_pct": score,
    "gate": gate,
    "checks": result_pdf.to_dict(orient="records"),
}

with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"Wrote {report_path}")

Wrote ../quality_reports/techmart_20260509T095419Z.json


## 9. Push the score to InfluxDBCloses the loop with the observability dashboard. Cell is wrapped in a `try` so it still runs without InfluxDB.

In [10]:
try:
    from influxdb_client import (
        InfluxDBClient,
        Point,
        WritePrecision
    )

    from influxdb_client.client.write_api import (
        SYNCHRONOUS
    )

    URL = os.environ.get(
        "INFLUXDB_URL",
        "http://influxdb:8086"
    )

    TOKEN = "my-super-secret-admin-token"
    ORG = "bdarch"
    BUCKET = "pipeline_metrics"

    client = InfluxDBClient(
        url=URL,
        token=TOKEN,
        org=ORG
    )

    write_api = client.write_api(
        write_options=SYNCHRONOUS
    )

    point = (
        Point("pipeline_metrics")
        .tag("component", "data_quality")
        .tag("scenario", "techmart")
        .tag("team", "techmart")
        .field("quality_score", float(score))
        .field("checks_passed", int(passed))
        .field("checks_failed", int(failed))
        .time(
            datetime.datetime.utcnow(),
            WritePrecision.S
        )
    )

    write_api.write(
        bucket=BUCKET,
        org=ORG,
        record=point
    )

    client.close()

    print(
        f"Pushed quality_score={score}% "
        f"to InfluxDB at {URL}"
    )

except Exception as e:
    print(f"InfluxDB write skipped: {e}")

Pushed quality_score=100.0% to InfluxDB at http://influxdb:8086


## 10. What we learned

- The single highest-leverage check is **O-8**  
  (`total_amount = quantity × unit_price`).
  It catches the entire pricing-bug class.

- Allow-lists (O-9, O-10) beat regexes for enums —
  the upstream service either knows the value or it does not.

- Tying the score to InfluxDB means the next step
  (drift detection on `quality_score`)
  is one Grafana panel away.

### Out of scope for this iteration

- No products/customers checks.  
  We treat them as v3 work.

- No streaming integration.  
  This notebook runs against a batch CSV snapshot;
  the same checks would run unchanged inside
  Spark Structured Streaming because PyDeequ runs
  on the same DataFrame API.

- No ML-based anomaly detection on metrics.  
  We stop at static thresholds,
  which the rubric considers sufficient.

In [11]:
spark.stop()